# Kazakh Profanity Detection — KBTU DSSC5113

This notebook runs the supplied saved model without retraining and optionally reproduces the complete training pipeline. CPU is sufficient.

**Data:** 1,400 real reviews with AI annotations + 600 explicitly marked synthetic examples. No independent human gold-standard annotation. Full source and policy: `docs/DATA_CARD.md` and `docs/ANNOTATION_POLICY.md`.

**Main result:** new held-out test F1 0.8571 (205 examples); new real-only F1 0.9565 (145 examples). These are not representative marketplace deployment estimates.

In Colab, upload the **complete project ZIP** when the first cell prompts. If you cloned the repository into Colab, change directory to its root first. Locally, launch Jupyter from the project root.


In [ ]:
from pathlib import Path
import os, sys, json, io, zipfile, subprocess

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and not Path('data/base_annotations.csv').exists():
    print('Upload the complete Kazakh_Profanity_Detection_Project.zip')
    uploaded = files.upload()
    archives = [name for name in uploaded if name.lower().endswith('.zip')]
    if len(archives) != 1:
        raise ValueError('Upload exactly one project ZIP.')
    destination = Path('/content').resolve()
    with zipfile.ZipFile(io.BytesIO(uploaded[archives[0]])) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination):
                raise ValueError('Unsafe archive path')
        archive.extractall(destination)
    os.chdir(destination / 'kazakh_profanity_project')

PROJECT_ROOT = Path.cwd().resolve()
assert (PROJECT_ROOT / 'src/predict.py').exists(), 'Run from the project root.'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Project:', PROJECT_ROOT)


## Install and verify dependencies

Fresh Colab installs the pinned requirements. The tested environment is Python 3.12.14. If Colab asks you to restart after installation, restart and rerun the notebook from the first cell.

In [ ]:
if IN_COLAB:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])

from importlib.metadata import version
for line in Path('requirements.txt').read_text().splitlines():
    if '==' in line:
        package, expected = line.split('==')
        actual = version(package)
        print(f'{package}: {actual}')
        assert actual == expected, f'Install requirements.txt; expected {package}=={expected}'


## Inspect data and fixed splits

The original sentiment labels were not reused. Explicit contrast pairs and detected near-duplicates stay in the same split. Predictive TF-IDF learns vocabulary/IDF only from training text.

In [ ]:
import pandas as pd
data = pd.read_csv('data/labeled.csv', keep_default_na=False)
print(data.groupby(['split', 'is_synthetic', 'label']).size().to_string())
print('Annotation methods:')
print(data.annotation_method.value_counts().to_string())
print('First three examples:')
print(data[['id', 'text', 'label', 'is_synthetic']].head(3).to_string(index=False))


## Load the saved model and enter your own text

Edit `custom_text` and rerun. No training is needed. The probability is a model estimate, not calibrated marketplace prevalence. The output also gives a logit/decision score.

In [ ]:
from src.predict import predict

custom_text = "Тауар сапасы жақсы, жеткізу жылдам."
prediction = predict(custom_text, explain=True)
print(json.dumps(prediction, ensure_ascii=False, indent=2))


{
  "prediction": 0,
  "profanity_probability": 0.0902420338854179,
  "decision_score": -2.3106832667119725,
  "threshold": 0.4,
  "normalized_text": "тауар сапасы жақсы, жеткізу жылдам.",
  "top_ngram_contributions": [
    {
      "ngram": "та",
      "logit_contribution": 0.09777613828442852
    },
    {
      "ngram": "сы",
      "logit_contribution": -0.08161958914310727
    },
    {
      "ngram": "қсы",
      "logit_contribution": -0.06860660397340339
    },
    {
      "ngram": "жақ",
      "logit_contribution": -0.06812256993191186
    },
    {
      "ngram": "қс",
      "logit_contribution": -0.0681002572218128
    },
    {
      "ngram": "жақс",
      "logit_contribution": -0.06799248101287235
    },
    {
      "ngram": "ақсы",
      "logit_contribution": -0.06728794928681237
    },
    {
      "ngram": "жақсы",
      "logit_contribution": -0.06724866450986527
    },
    {
      "ngram": "ақс",
      "logit_contribution": -0.06606834644556539
    },
    {
      "ngram": " жа

## Recorded held-out evaluation

This reads the saved evaluation instead of recomputing it. Rows of the confusion matrix are true labels; columns are predicted labels. Class order is [0, 1].

In [ ]:
all_metrics = json.loads(Path('results/metrics.json').read_text())
metrics = all_metrics['subgroups']['new_test_only']
print('Primary scope: new examples only; original 83 test examples remain held out and are reported separately.')
for key in ['n', 'positive_count', 'precision', 'recall', 'f1', 'confusion_matrix']:
    print(key, ':', metrics[key])
print('Real-only:', all_metrics['subgroups']['new_real_only'])
print('Real marketplace:', all_metrics['subgroups']['real_marketplace'])

import matplotlib.pyplot as plt
image = plt.imread('results/confusion_matrix_new_test.png')
plt.figure(figsize=(7, 6))
plt.imshow(image)
plt.axis('off')
plt.show()


## Inspect actual mistakes

The new test contains 21 mistakes. Examples include clean contact-list complaints flagged as profane, obscene abbreviations in positive-tone reviews missed, and correlated failures in held-out synthetic template families. See `results/ERROR_ANALYSIS.md` for local feature contributions and improvements. No settings were changed after inspecting these errors.

In [ ]:
errors = pd.read_csv('results/errors.csv')
print(errors[['id', 'text', 'label', 'prediction', 'probability']].to_string(index=False))


## Optional: reproduce preparation and training

Set `RUN_TRAINING=True`. This uses fixed included annotations, fits on train, selects on validation, then evaluates the frozen reproduction in a separate folder. It does not call GPT or need the entire original corpus. Do not use observed test results to select new settings and still call the same test untouched.

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    from src.prepare_data import prepare
    from src.train import train
    from src.evaluate import evaluate
    prepare()
    train('artifacts/retrained')
    if Path('results/retrained/metrics.json').exists():
        print('Reproduction results already exist; reading them without overwriting.')
        reproduced = json.loads(Path('results/retrained/metrics.json').read_text())
    else:
        reproduced = evaluate('artifacts/retrained/model.joblib', 'results/retrained')
    print(reproduced)
else:
    print('Using supplied saved model. Set RUN_TRAINING=True to reproduce training.')


## Verification and defense

These tests check split separation, preprocessing consistency, annotation edge handling, and saved-model inference. They do not establish linguistic correctness of AI labels. Review the annotation policy and `docs/DEFENSE_GUIDE.md` before your individual defense.

In [ ]:
subprocess.check_call([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'])


## Limitations and acknowledgment

- AI annotations were not independently checked by a human; the data can contain annotation mistakes.
- Candidate selection favors known profanity families. The benchmark has selected class prevalence.
- Synthetic pairs are easier than many real reviews; the subgroup scores differ.
- Real test positives are appstore reviews. The new 408 synthetic additions are templated; all product substitutions from each template share a split. The physical marketplace subset has zero positives and cannot measure profanity recall.
- The assistant prepared this draft code, annotations, constructed examples, training, and documentation. Record your actual student contribution honestly in `SUBMISSION_CHECKLIST.md`.
